In [2]:
import pandas as pd
import requests
import re
from io import StringIO


# --------- Helper: Extract numeric value ----------
def extract_number(value):
    if pd.isna(value):
        return None
    match = re.search(r"[-+]?\d*\.?\d+", str(value))
    return float(match.group()) if match else None


# --------- Main Scraper Function ----------
def scrape_city_climate(city_name):
    
    print(f"Scraping {city_name}...")
    
    url = f"https://en.wikipedia.org/wiki/{city_name.replace(' ', '_')}"
    
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to fetch {city_name}")
        return None
    
    tables = pd.read_html(StringIO(response.text))
    
    # ---- Find climate table dynamically ----
    climate_table = None
    
    for table in tables:
        table_str = table.astype(str)
        if table_str.apply(lambda x: x.str.contains("rain", case=False, na=False)).any().any():
            climate_table = table
            break
    
    if climate_table is None:
        print(f"No climate table found for {city_name}")
        return None
    
    # ---- Extract rows ----
    rainfall = climate_table[climate_table.iloc[:,0].astype(str).str.contains("rain", case=False, na=False)]
    humidity = climate_table[climate_table.iloc[:,0].astype(str).str.contains("humidity", case=False, na=False)]
    temperature = climate_table[climate_table.iloc[:,0].astype(str).str.contains("Daily mean", case=False, na=False)]
    
    if rainfall.empty or humidity.empty or temperature.empty:
        print(f"Incomplete climate data for {city_name}")
        return None
    
    rainfall = rainfall.iloc[:, :14]
    humidity = humidity.iloc[:, :14]
    temperature = temperature.iloc[:, :14]
    
    rain_values = [extract_number(v) for v in rainfall.iloc[0, 1:13].values]
    hum_values = [extract_number(v) for v in humidity.iloc[0, 1:13].values]
    temp_values = [extract_number(v) for v in temperature.iloc[0, 1:13].values]
    
    months = ["Jan","Feb","Mar","Apr","May","Jun",
              "Jul","Aug","Sep","Oct","Nov","Dec"]
    
    df = pd.DataFrame({
        "City": city_name,
        "Month": months,
        "Temperature_C": temp_values,
        "Rainfall_mm": rain_values,
        "Humidity_%": hum_values
    })
    
    print(f"{city_name} done ✅")
    
    return df

 # Cities For Scarping Climate Metrics

In [3]:
cities = ["Bhubaneswar", "Mumbai", "Delhi", "Chennai", "Kolkata"]

dfs = []

for city in cities:
    df = scrape_city_climate(city)
    if df is not None:
        dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)


Scraping Bhubaneswar...
Bhubaneswar done ✅
Scraping Mumbai...
Mumbai done ✅
Scraping Delhi...
Delhi done ✅
Scraping Chennai...
Chennai done ✅
Scraping Kolkata...
Kolkata done ✅


In [4]:
all_data.head(50)

,City,Month,Temperature_C,Rainfall_mm,Humidity_%
0,Bhubaneswar,Jan,22.2,13.1,55.0
1,Bhubaneswar,Feb,25.6,21.1,52.0
2,Bhubaneswar,Mar,29.2,20.6,58.0
3,Bhubaneswar,Apr,31.4,40.4,64.0
4,Bhubaneswar,May,32.0,101.6,67.0
5,Bhubaneswar,Jun,30.7,208.5,75.0
6,Bhubaneswar,Jul,29.0,359.7,85.0
7,Bhubaneswar,Aug,28.6,374.6,86.0
8,Bhubaneswar,Sep,28.8,281.7,85.0
9,Bhubaneswar,Oct,27.7,201.2,80.0


# Soil Types fo different cities of india

In [5]:
soil_map = {
    # North
    "Delhi": "Alluvial",
    "Lucknow": "Alluvial",
    "Chandigarh": "Alluvial",
    "Jaipur": "Desert",
    "Dehradun": "Alluvial",

    # West
    "Mumbai": "Laterite",
    "Pune": "Black",
    "Ahmedabad": "Black",
    "Surat": "Black",
    "Jodhpur": "Desert",

    # South
    "Chennai": "Red",
    "Bengaluru": "Red",
    "Hyderabad": "Red",
    "Kochi": "Laterite",
    "Mysuru": "Red",

    # East
    "Kolkata": "Alluvial",
    "Bhubaneswar": "Laterite",
    "Ranchi": "Red",
    "Patna": "Alluvial",
    "Guwahati": "Alluvial",

    # Central
    "Bhopal": "Black",
    "Nagpur": "Black",
    "Raipur": "Red",
    "Indore": "Black",
    "Jabalpur": "Black"
}
cities = list(soil_map.keys())

In [6]:
def build_india_climate_dataset(soil_map):
    
    print("Building full India climate dataset...\n")
    
    cities = list(soil_map.keys())
    dfs = []
    
    for city in cities:
        df = scrape_city_climate(city)
        
        if df is not None:
            df["Soil_Type"] = soil_map[city]
            dfs.append(df)
        else:
            print(f"Skipping {city}\n")
    
    if not dfs:
        print("No data collected.")
        return None
    
    final_df = pd.concat(dfs, ignore_index=True)
    
    print("\nDataset build complete ✅")
    print(f"Total Rows: {final_df.shape[0]}")
    
    return final_df

In [7]:
all_data = build_india_climate_dataset(soil_map)

all_data.head()

Building full India climate dataset...

Scraping Delhi...
Delhi done ✅
Scraping Lucknow...
Lucknow done ✅
Scraping Chandigarh...
Chandigarh done ✅
Scraping Jaipur...
Jaipur done ✅
Scraping Dehradun...
Dehradun done ✅
Scraping Mumbai...
Mumbai done ✅
Scraping Pune...
Pune done ✅
Scraping Ahmedabad...
Ahmedabad done ✅
Scraping Surat...
Surat done ✅
Scraping Jodhpur...
Jodhpur done ✅
Scraping Chennai...
Chennai done ✅
Scraping Bengaluru...
Bengaluru done ✅
Scraping Hyderabad...
Hyderabad done ✅
Scraping Kochi...
Incomplete climate data for Kochi
Skipping Kochi

Scraping Mysuru...
Incomplete climate data for Mysuru
Skipping Mysuru

Scraping Kolkata...
Kolkata done ✅
Scraping Bhubaneswar...
Bhubaneswar done ✅
Scraping Ranchi...
Incomplete climate data for Ranchi
Skipping Ranchi

Scraping Patna...
Incomplete climate data for Patna
Skipping Patna

Scraping Guwahati...
Guwahati done ✅
Scraping Bhopal...
Bhopal done ✅
Scraping Nagpur...
Nagpur done ✅
Scraping Raipur...
Raipur done ✅
Scraping In

,City,Month,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type
0,Delhi,Jan,13.8,19.1,57.0,Alluvial
1,Delhi,Feb,17.4,21.3,46.0,Alluvial
2,Delhi,Mar,22.7,17.4,37.0,Alluvial
3,Delhi,Apr,28.9,16.3,25.0,Alluvial
4,Delhi,May,32.7,30.7,28.0,Alluvial


In [8]:
city_avg = all_data.groupby("City").agg({
    "Temperature_C": "mean",
    "Rainfall_mm": "mean",
    "Humidity_%": "mean",
    "Soil_Type": "first"
}).reset_index()

city_avg.head(50)

,City,Temperature_C,Rainfall_mm,Humidity_%,Soil_Type
0,Ahmedabad,27.808333,65.300000,41.250000,Black
1,Bengaluru,24.608333,89.816667,51.750000,Red
2,Bhopal,25.533333,91.750000,44.666667,Black
3,Bhubaneswar,27.708333,138.141667,69.666667,Laterite
4,Chandigarh,24.241667,86.533333,51.500000,Alluvial
5,Chennai,29.083333,117.366667,68.833333,Red
6,Dehradun,22.091667,180.125000,61.333333,Alluvial
7,Delhi,25.208333,64.533333,48.750000,Alluvial
8,Guwahati,24.875000,141.283333,72.333333,Alluvial
9,Hyderabad,27.016667,71.633333,47.500000,Red


In [9]:
city_avg.shape

(21, 5)

In [13]:
import os

os.makedirs("Data", exist_ok=True)

all_data.to_csv("Data/india_climate_soil_dataset.csv", index=False)